# Kia Soul: Decision Tree Baseline

This notebook trains a first model to classify CAN messages as normal (0) or attack (1). It uses the Kia Soul dataset and tests on an attack type the model never saw during training.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

## Load the data

The Parquet file is in the project root. Each row represents one CAN message.

In [2]:
df = pd.read_parquet("../../kia_soul_features_all.parquet")

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (4342470, 24)


,timestamp,can_id,can_id_dec,dlc,byte_0,byte_1,byte_2,byte_3,byte_4,byte_5,...,bus_count_100ms,label,label_bin,vehicle,attack_type,source_file,source_folder,recording_id,split,round
0,1.513925e+09,0081,129,8,127,132,148.0,0.0,0.0,0.0,...,1,R,0,Kia_Soul,mixed/unknown,File_2.csv,car_track_final_1st_release,car_track_final_1st_release/File_2.csv,test,final_1st
1,1.513925e+09,018F,399,8,254,118,49.0,0.0,0.0,70.0,...,2,R,0,Kia_Soul,mixed/unknown,File_2.csv,car_track_final_1st_release,car_track_final_1st_release/File_2.csv,test,final_1st
2,1.513925e+09,02A0,672,8,194,0,148.0,157.0,198.0,17.0,...,3,R,0,Kia_Soul,mixed/unknown,File_2.csv,car_track_final_1st_release,car_track_final_1st_release/File_2.csv,test,final_1st
3,1.513925e+09,0329,809,8,15,179,127.0,140.0,17.0,49.0,...,4,R,0,Kia_Soul,mixed/unknown,File_2.csv,car_track_final_1st_release,car_track_final_1st_release/File_2.csv,test,final_1st
4,1.513925e+09,0382,898,8,64,254,15.0,0.0,0.0,0.0,...,5,R,0,Kia_Soul,mixed/unknown,File_2.csv,car_track_final_1st_release,car_track_final_1st_release/File_2.csv,test,final_1st


## Check the labels

`label_bin` is the correct answer: 0 means normal and 1 means attack. These counts include the complete dataset.

In [3]:
class_counts = df["label_bin"].value_counts().sort_index()

print("Message counts:")
print(class_counts)

Message counts:
label_bin
0    3673516
1     668954
Name: count, dtype: int64


## Load the prepared splits

`splits/make_kia_splits.py` already prepared the splits. Each one lists row positions in the Parquet file, so the SHA-256 check makes sure the file is the one the splits were made for.

In [4]:
splits_dir = Path("../../splits/generated/kia_soul")
manifest = json.loads((splits_dir / "manifest.json").read_text())

with Path("../../kia_soul_features_all.parquet").open("rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == manifest["sha256"], \
        "Parquet changed since the splits were generated"

## Train and test on each held-out attack

In `unseen_dos`, `unseen_injection`, `unseen_spoofing` and `unseen_replay`, one attack type is removed from training and used only for testing. The `official` fold is skipped: the official test data stays unused.

Inputs are the message identifier, its length and the eight byte columns. Missing bytes of short messages are filled with -1, a value no real byte can have. The tree depth is limited to six.

In [5]:
features = manifest["variants"]["standard"]

for fold in manifest["folds"]:
    if fold["name"] == "official":
        continue

    train = df.iloc[np.load(splits_dir / fold["train"]["indices"])]
    test = df.iloc[np.load(splits_dir / fold["test"]["indices"])]

    model = DecisionTreeClassifier(max_depth=6, random_state=42)
    model.fit(train[features].fillna(-1), train["label_bin"])
    predictions = model.predict(test[features].fillna(-1))

    print("===", fold["name"], "===")
    print(confusion_matrix(test["label_bin"], predictions))
    print(classification_report(test["label_bin"], predictions,
                                target_names=["Normal", "Attack"], zero_division=0))

=== unseen_dos ===
[[103928      0]
 [ 16072      0]]
              precision    recall  f1-score   support

      Normal       0.87      1.00      0.93    103928
      Attack       0.00      0.00      0.00     16072

    accuracy                           0.87    120000
   macro avg       0.43      0.50      0.46    120000
weighted avg       0.75      0.87      0.80    120000



=== unseen_injection ===
[[462399    437]
 [ 64451  76713]]


              precision    recall  f1-score   support

      Normal       0.88      1.00      0.93    462836
      Attack       0.99      0.54      0.70    141164

    accuracy                           0.89    604000
   macro avg       0.94      0.77      0.82    604000
weighted avg       0.90      0.89      0.88    604000



=== unseen_spoofing ===
[[708791    117]
 [ 34533  37447]]


              precision    recall  f1-score   support

      Normal       0.95      1.00      0.98    708908
      Attack       1.00      0.52      0.68     71980

    accuracy                           0.96    780888
   macro avg       0.98      0.76      0.83    780888
weighted avg       0.96      0.96      0.95    780888



=== unseen_replay ===
[[410698      0]
 [155640      0]]


              precision    recall  f1-score   support

      Normal       0.73      1.00      0.84    410698
      Attack       0.00      0.00      0.00    155640

    accuracy                           0.73    566338
   macro avg       0.36      0.50      0.42    566338
weighted avg       0.53      0.73      0.61    566338

